# 🤖 Google Colab Backend Server for Enquiry Bot
This notebook sets up a **FastAPI** server with an **Ngrok HTTPS Tunnel** on Google Colab, allowing your local **Enquiry Bot Web App** to connect directly to Google Colab for AI inference!

### Step 1: Install Dependencies
Run this cell to install FastAPI, Uvicorn, PyNgrok, and Pydantic.

In [ ]:
!pip install -q fastapi uvicorn pyngrok pydantic requests

### Step 2: Set Ngrok Auth Token (Optional but Recommended)
Get your free token from [https://dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken).

In [ ]:
from pyngrok import ngrok

# Replace with your Ngrok Authtoken if available
NGROK_TOKEN = "YOUR_NGROK_AUTHTOKEN_HERE"

if NGROK_TOKEN and NGROK_TOKEN != "YOUR_NGROK_AUTHTOKEN_HERE":
    ngrok.set_auth_token(NGROK_TOKEN)
    print("✅ Ngrok Authtoken configured!")
else:
    print("ℹ️ Running without custom Ngrok Authtoken.")

### Step 3: Define FastAPI Inference Server Code

In [ ]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import time

app = FastAPI(title="Colab Enquiry Bot Backend")

# Enable CORS for web frontend
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class QueryRequest(BaseModel):
    prompt: Optional[str] = None
    query: Optional[str] = None
    session_id: Optional[str] = "colab-session"

@app.get("/")
def root():
    return {"status": "online", "message": "Google Colab Enquiry Server Ready!"}

@app.get("/health")
def health():
    return {"status": "healthy", "colab": True}

@app.post("/query")
def process_query(req: QueryRequest):
    user_text = (req.prompt or req.query or "").strip()
    start_time = time.time()
    
    # You can plug in HuggingFace Transformers, Ollama, Gemini API, or custom LLM here!
    answer = f"⚡ [Google Colab LLM]: I have processed your inquiry: '{user_text}'. Our AI model running live on Google Colab hardware is ready to assist you!"
    
    latency = (time.time() - start_time) * 1000
    return {
        "answer": answer,
        "source": "colab_llm",
        "latencyMs": round(latency, 2)
    }

### Step 4: Launch Public HTTPS Tunnel and Server
Run this cell to launch the server and get your HTTPS URL!

In [ ]:
import nest_asyncio
nest_asyncio.apply()

# Open Ngrok Tunnel on Port 8000
public_url = ngrok.connect(8000).public_url
print("=" * 65)
print("https://colab.research.google.com/drive/14u59WJbVveBa4zEMoqx7VGzpRv6m91Nz?usp=sharing")
print(public_url)
print("=" * 65)
print("📋 Copy the URL above and paste it into the 'Colab Settings' tab of your Enquiry Bot App!")

uvicorn.run(app, port=8000)